In [1]:
!pip install requests beautifulsoup4 lxml pandas


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
import sqlite3

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
# Step 1: Define the target URL (e.g., a data science blog listing)
url = "https://towardsdatascience.com/"
# Step 2: Define a User-Agent header – essential for ethical and often successful scraping
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}
# Step 3: Send the GET request with the specified headers
response = requests.get(url, headers=headers)
# Step 4: Verify the server's response status. A 200 code is a success!
print("--- Initiating Contact ---")
print(f"Status Code: {response.status_code}")
# Step 5: Get a peek at the raw HTML received.
print("HTML Content Preview (First 500 characters):")
print(response.text[:500])
print("---")

--- Initiating Contact ---
Status Code: 200
HTML Content Preview (First 500 characters):
<!DOCTYPE html>
<html lang="en-US">
<head>
	<meta charset="UTF-8" />
	<script src="https://h030.towardsdatascience.com/script.js"></script><!-- Google Tag Manager -->
<script>
	(function (w, d, s, l, i) {
		w[l] = w[l] || [];
		w[l].push({
			'gtm.start': new Date().getTime(),
			event: 'gtm.js'
		});
		var f = d.getElementsByTagName(s)[0],
			j = d.createElement(s),
			dl = l != 'dataLayer' ? '&l=' + l : '';
		j.async = true;
		j.src =
			'https://www.googletagmanager.com/gtm.js?id=' + i + dl;

---


In [4]:
import requests
from bs4 import BeautifulSoup

# Step 1: Setup the URL and "Human" Headers
url = "https://towardsdatascience.com/"
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5'
}

# Step 2: Make the request
try:
    response = requests.get(url, headers=headers)
    response.raise_for_status() 
    html_content = response.text
    print(f"--- Connection Successful! Status Code: {response.status_code} ---")

    # Step 3: Transform raw HTML into a BeautifulSoup object
    soup = BeautifulSoup(html_content, 'lxml')

    # Step 4: Confirm by printing the Page Title
    if soup.title:
        print(f"Parsed Page Title: {soup.title.text.strip()}")
    else:
        print("Page title not found. HTML parsing might need adjustment.")

except requests.exceptions.HTTPError as err:
    print(f"HTTP Error occurred: {err}")
    print("Tip: If you see '403 Forbidden', the site is blocking this script.")

--- Connection Successful! Status Code: 200 ---
Parsed Page Title: Towards Data Science


In [5]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

url = "https://en.wikipedia.org/wiki/Main_Page"
# This must be on ONE line to avoid the 'unterminated string literal' error
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}

print(f"Attempting to scrape: {url}")

try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status() 
    
    soup = BeautifulSoup(response.text, 'lxml')
    article_links_data = []

    # 1. Target the "In the news" section (id='mp-itn')
    in_the_news_div = soup.find('div', id='mp-itn')
    if in_the_news_div:
        print("\n--- From 'In the news' section ---")
        links = in_the_news_div.find_all('a', href=True)
        for link in links:
            title_text = link.text.strip()
            href = link['href']

            if title_text and href.startswith('/wiki/') and ':' not in href: 
                absolute_url = urljoin(url, href)
                if not any(data['url'] == absolute_url for data in article_links_data):
                    article_links_data.append({'title': title_text, 'url': absolute_url})

    # 2. Target the "On this day" section (id='mp-otd')
    on_this_day_div = soup.find('div', id='mp-otd')
    if on_this_day_div:
        print("\n--- From 'On this day' section ---")
        links = on_this_day_div.find_all('a', href=True)
        for link in links:
            title_text = link.text.strip()
            href = link['href']
            if title_text and href.startswith('/wiki/') and ':' not in href:
                absolute_url = urljoin(url, href)
                if not any(data['url'] == absolute_url for data in article_links_data):
                    article_links_data.append({'title': title_text, 'url': absolute_url})

    # Display results
    if article_links_data:
        print("\nExtracted Article Titles and URLs (First 10):")
        for i, article in enumerate(article_links_data[:10]):
            print(f"- Title: '{article['title']}'")
            print(f"  URL: {article['url']}")
    else:
        print("\nNo article links found. The HTML structure might have changed.")

except requests.exceptions.RequestException as e:
    print(f"\nError fetching URL {url}: {e}")
except Exception as e:
    print(f"\nAn unexpected error occurred: {e}")

print("\n--- Scraping Task Complete ---")

Attempting to scrape: https://en.wikipedia.org/wiki/Main_Page

--- From 'In the news' section ---

--- From 'On this day' section ---

Extracted Article Titles and URLs (First 10):
- Title: 'Tenerife'
  URL: https://en.wikipedia.org/wiki/Tenerife
- Title: 'a hantavirus outbreak'
  URL: https://en.wikipedia.org/wiki/MV_Hondius_hantavirus_outbreak
- Title: 'Canvas'
  URL: https://en.wikipedia.org/wiki/Instructure
- Title: 'goes offline'
  URL: https://en.wikipedia.org/wiki/2026_Canvas_security_incident
- Title: 'ransomware'
  URL: https://en.wikipedia.org/wiki/Ransomware
- Title: 'ShinyHunters'
  URL: https://en.wikipedia.org/wiki/ShinyHunters
- Title: 'Ted Turner'
  URL: https://en.wikipedia.org/wiki/Ted_Turner
- Title: 'Wu Yize'
  URL: https://en.wikipedia.org/wiki/Wu_Yize
- Title: 'Shaun Murphy'
  URL: https://en.wikipedia.org/wiki/Shaun_Murphy
- Title: 'the World Snooker Championship'
  URL: https://en.wikipedia.org/wiki/2026_World_Snooker_Championship

--- Scraping Task Complete ---

In [6]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

# We need to get the page again to find the table
url = "https://en.wikipedia.org/wiki/List_of_brightest_stars" # Example page with a table
headers = {'User-Agent': 'Mozilla/5.0'}
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, 'lxml')

print("--- PROCEDURE 6: Extract Data from HTML Tables ---")
target_table = soup.find('table', class_='wikitable')

if target_table:
    table_headers = [th.text.strip() for th in target_table.find('tr').find_all('th')]
    table_data = []
    
    for row in target_table.find_all('tr')[1:6]: 
        cells = row.find_all(['td', 'th'])
        cell_texts = [cell.text.strip() for cell in cells]
        table_data.append(cell_texts)

    if table_data:
        df = pd.DataFrame(table_data) # Simplified for the lab
        print("\nDataFrame Representation:")
        print(df.head())
else:
    print("Table not found. Try a page with a 'wikitable' class.")

--- PROCEDURE 6: Extract Data from HTML Tables ---

DataFrame Representation:
             0
0  O-type star
1  B-type star
2  A-type star
3  F-type star
4  G-type star


In [7]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import time

print("--- PROCEDURE 7: Handle Pagination (Quotes to Scrape) ---")
start_url = "http://quotes.toscrape.com/"
all_quotes = [] 
pages_to_collect = 3 
current_page_url = start_url
page_counter = 0

while page_counter < pages_to_collect:
    print(f"\nAttempting to fetch page {page_counter + 1}: {current_page_url}")
    try:
        # FIXED: Kept on one line to avoid SyntaxError
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
        
        response = requests.get(current_page_url, headers=headers, timeout=10)
        response.raise_for_status() 
        soup = BeautifulSoup(response.text, 'lxml') 

        # Step 1: Extract data
        quote_elements = soup.find_all('div', class_='quote')
        if quote_elements:
            for quote_item in quote_elements:
                text = quote_item.find('span', class_='text').text.strip()
                author = quote_item.find('small', class_='author').text.strip()
                all_quotes.append({'text': text, 'author': author})
            print(f"Collected {len(quote_elements)} quotes from this page.")
        else:
            print("No quote elements found. Stopping.")
            break 

        # Step 2: Locate the "Next Page" link
        next_li = soup.find('li', class_='next')
        if next_li:
            next_link_element = next_li.find('a', href=True)
            if next_link_element:
                next_page_relative_url = next_link_element['href']
                current_page_url = urljoin(start_url, next_page_relative_url)
                page_counter += 1
                time.sleep(1) # Be polite!
            else:
                break
        else:
            print("No 'next' link found. Assuming last page.")
            break 

    except Exception as e:
        print(f"ERROR: {e}")
        break

print(f"\n--- Total Quotes Collected Across {page_counter + 1} pages: {len(all_quotes)} ---")
for i, quote in enumerate(all_quotes[:5]):
    print(f"- \"{quote['text']}\" - {quote['author']}")
print("--- End of Procedure ---")

--- PROCEDURE 7: Handle Pagination (Quotes to Scrape) ---

Attempting to fetch page 1: http://quotes.toscrape.com/
Collected 10 quotes from this page.

Attempting to fetch page 2: http://quotes.toscrape.com/page/2/
Collected 10 quotes from this page.

Attempting to fetch page 3: http://quotes.toscrape.com/page/3/
Collected 10 quotes from this page.

--- Total Quotes Collected Across 4 pages: 30 ---
- "“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”" - Albert Einstein
- "“It is our choices, Harry, that show what we truly are, far more than our abilities.”" - J.K. Rowling
- "“There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”" - Albert Einstein
- "“The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.”" - Jane Austen
- "“Imperfection is beauty, madness is genius and it's better to be absolutely

In [8]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd

url = "https://en.wikipedia.org/wiki/Comparison_of_deep_learning_software"
headers = {'User-Agent': 'Mozilla/5.0'}

print("--- PROCEDURE 8: Clean and Standardize Extracted Data (Revised for Wikipedia Table) ---")

try:
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'lxml')
    table = soup.find('table', class_='wikitable')

    if table:
        header_cells = table.find('tr').find_all('th')
        headers_text = [th.text.strip() for th in header_cells]

        # Define the exact headers we now expect
        software_idx = headers_text.index('Software') if 'Software' in headers_text else -1
        creator_idx = headers_text.index('Creator') if 'Creator' in headers_text else -1
        initial_release_idx = headers_text.index('Initial release') if 'Initial release' in headers_text else -1

        if any(idx == -1 for idx in [software_idx, creator_idx, initial_release_idx]):
            print("ERROR: One or more required columns are missing.")
            print(f"Found headers: {headers_text}")
        else:
            cleaned_data_records = []
            data_rows = table.find_all('tr')[1:] 

            for row_num, row in enumerate(data_rows):
                cols = row.find_all('td')
                if len(cols) > max(software_idx, creator_idx, initial_release_idx):
                    software_name = cols[software_idx].text.strip()
                    creator_name = cols[creator_idx].text.strip()
                    raw_initial_release = cols[initial_release_idx].text.strip()

                    # Clean brackets like [1] using Regular Expressions (Regex)
                    clean_initial_release = re.sub(r'\[.*?\]', '', raw_initial_release).strip()
                    
                    cleaned_data_records.append({
                        'Software': software_name,
                        'Creator': creator_name,
                        'Initial_Release_Date': clean_initial_release
                    })

            # Display the results
            df_cleaned = pd.DataFrame(cleaned_data_records)
            print("\nDataFrame Head (Cleaned Data):")
            print(df_cleaned.head().to_string())
    else:
        print("Table not found. Check the URL or table class.")

except Exception as e:
    print(f"An error occurred: {e}")
finally:
    print("--- End of Procedure ---")

--- PROCEDURE 8: Clean and Standardize Extracted Data (Revised for Wikipedia Table) ---

DataFrame Head (Cleaned Data):
         Software                                                                     Creator Initial_Release_Date
0           BigDL                                                           Jason Dai (Intel)                 2016
1           Caffe                                         Berkeley Vision and Learning Center                 2013
2         Chainer                                                          Preferred Networks                 2015
3  Deeplearning4j  Skymind engineering team; Deeplearning4j community; originally Adam Gibson                 2014
4       DeepSpeed                                                                   Microsoft                 2019
--- End of Procedure ---


In [9]:
import requests
from bs4 import BeautifulSoup

print("--- PROCEDURE 9: Implement Error Handling and Robustness ---")

# Test URLs designed to trigger different error types
test_scenarios = {
    "valid_article_page": "https://en.wikipedia.org/wiki/Artificial_intelligence",
    "non_existent_article": "http://httpbin.org/status/404",
    "slow_server_sim": "http://httpbin.org/delay/6",
    "invalid_domain_connect": "http://this.domain.does.not.resolve.abc"
}

for scenario_name, test_url in test_scenarios.items():
    print(f"\nTesting Scenario: '{scenario_name}' at {test_url}")
    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        # We set a 5-second timeout. If the server takes 6 seconds (like our test), it triggers an error.
        response = requests.get(test_url, headers=headers, timeout=5)

        # Raise an exception for HTTP error codes (4xx, 5xx)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.text, 'lxml')

        if scenario_name == "valid_article_page":
            main_title_element = soup.find('h1', id='firstHeading')
        else:
            main_title_element = soup.find('h1')

        if main_title_element:
            print(f"SUCCESS: Fetched and found title: '{main_title_element.text.strip()[:100]}...'")
        else:
            print("SUCCESS (with caveat): Fetched page, but main title element not found.")

    except requests.exceptions.HTTPError as e:
        print(f"ERROR (HTTP): Status Code {e.response.status_code} for {test_url}. Reason: {e.response.reason}")
    except requests.exceptions.ConnectionError as e:
        print(f"ERROR (Connection): Could not resolve {test_url}.")
    except requests.exceptions.Timeout as e:
        print(f"ERROR (Timeout): Request to {test_url} timed out after 5 seconds.")
    except Exception as e:
        print(f"ERROR (Unexpected): {e}")

print("\n--- All Scenarios Tested ---")

--- PROCEDURE 9: Implement Error Handling and Robustness ---

Testing Scenario: 'valid_article_page' at https://en.wikipedia.org/wiki/Artificial_intelligence
SUCCESS: Fetched and found title: 'Artificial intelligence...'

Testing Scenario: 'non_existent_article' at http://httpbin.org/status/404
ERROR (Connection): Could not resolve http://httpbin.org/status/404.

Testing Scenario: 'slow_server_sim' at http://httpbin.org/delay/6
ERROR (Connection): Could not resolve http://httpbin.org/delay/6.

Testing Scenario: 'invalid_domain_connect' at http://this.domain.does.not.resolve.abc
ERROR (Connection): Could not resolve http://this.domain.does.not.resolve.abc.

--- All Scenarios Tested ---


In [10]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
import os 
import re 
import sqlite3 

print("\n--- PROCEDURE 10: Store the Extracted Data ---")

url = "https://en.wikipedia.org/wiki/Comparison_of_deep_learning_software"
headers = {'User-Agent': 'Mozilla/5.0'} 

try:
    # We use a 10-second timeout for a more stable connection
    response = requests.get(url, headers=headers, timeout=10) 
    response.raise_for_status() 
    soup = BeautifulSoup(response.text, 'lxml') 
    
    final_data_to_save = []
    table = soup.find('table', class_='wikitable') 
    
    if table:
        table_headers_raw = [th.text.strip() for th in table.find('tr').find_all('th')]
        # Clean headers: remove brackets, parentheses, and replace spaces with underscores
        table_headers = [re.sub(r'\[.*?\]|\s\(.*?\)', '', h).replace(' ', '_').strip() for h in table_headers_raw]
        
        data_rows = table.find_all('tr')[1:] 
        for i, row in enumerate(data_rows[:15]):
            cols = row.find_all('td')
            row_data = {}
            for j, col in enumerate(cols):
                if j < len(table_headers):
                    header_key = table_headers[j]
                    row_data[header_key] = col.text.strip()
            
            if row_data:
                final_data_to_save.append(row_data)

    if final_data_to_save:
        df_output = pd.DataFrame(final_data_to_save)
        print("\n--- Data Ready for Archiving (DataFrame Head) ---")
        print(df_output.head().to_string())

        # Step 2: Save as CSV
        csv_file_path = 'deep_learning_software_comparison.csv'
        df_output.to_csv(csv_file_path, index=False, encoding='utf-8')
        print(f"\nData saved to: {csv_file_path}")

        # Step 3: Save as JSON
        json_file_path = 'deep_learning_software_comparison.json'
        df_output.to_json(json_file_path, orient='records', indent=4)
        print(f"Data saved to: {json_file_path}")

        # Step 4: Save to SQLite Database
        db_file_path = 'deep_learning_software_comparison.db'
        conn = sqlite3.connect(db_file_path)
        df_output.to_sql('dl_software_comparison', conn, if_exists='replace', index=False)
        conn.close()
        print(f"Data saved to: {db_file_path} (Database)")

        print("\nVerification: CSV, JSON, and SQLite files are all present!")
    else:
        print("No data collected to save.")

except Exception as e:
    print(f"An error occurred: {e}")


--- PROCEDURE 10: Store the Extracted Data ---

--- Data Ready for Archiving (DataFrame Head) ---
         Software                                                                     Creator Initial_release Software_license Open_source                                         Platform         Written_in                                     Interface OpenMP_support        OpenCL_support CUDA_support ROCm_support Automatic_differentiation Has_pretrained_models Recurrent_nets Convolutional_nets RBM/DBNs Parallel_execution(multi_node) Actively_developed
0           BigDL                                                           Jason Dai (Intel)            2016       Apache 2.0         Yes                                     Apache Spark              Scala                                 Scala, Python                                                No           No                                             Yes            Yes                Yes                                               

In [11]:
# Fetching HTML Status
import requests
url = "https://en.wikipedia.org/wiki/Web_scraping"
headers = {'User-Agent': 'Mozilla/5.0'}

try:
    # Everything below 'try' must be indented
    response = requests.get(url, headers=headers, timeout=5)
    print(f"HTTP Status Code: {response.status_code}")
except requests.exceptions.RequestException as e:
    # Everything below 'except' must also be indented
    print(f"An error occurred: {e}")

HTTP Status Code: 200


In [12]:
# Parsing HTML and Getting Title Tag
import requests
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/Web_scraping"
headers = {'User-Agent': 'Mozilla/5.0'}

try:
    # Indented block starts here
    response = requests.get(url, headers=headers, timeout=5)
    response.raise_for_status()
    
    soup = BeautifulSoup(response.text, 'lxml')
    
    # We use .strip() to clean up any extra spaces in the title
    if soup.title:
        print(f"Page Title (from <title> tag): {soup.title.text.strip()}")
    else:
        print("No title tag found.")
        
except requests.exceptions.RequestException as e:
    print(f"Error: {e}")

Page Title (from <title> tag): Web scraping - Wikipedia
